# 공개용 작업 증빙 사본
[확실] 기존 파일의 코드·설명을 복사하고 출력·실행 번호·첨부 이미지를 제거했습니다. 개인 경로는 `private_inputs`로 바꿨습니다. 원자료 및 상세 결과는 포함하지 않습니다.
[확실] 이 사본을 새로 전체 실행한 기록은 아닙니다. 과거 실행 범위는 `evidence/verification.json`과 `docs/VALIDATION.md`를 확인하세요.


# 사회적 고립 예방을 위한 활동 저조 분석과 생활권 자원 연계
## 단계별 결과 통합 검토 노트북

[확실] 이 노트북은 기존 Step0~6에서 생성한 결과를 명시적인 경로로 읽고, 집계·시각화·산식 재현을 수행한다. 원자료부터 시설 질량·α 후보 탐색·AFT 학습을 모두 다시 실행하는 노트북은 아니다. 원본 입력·결과 파일을 변경하지 않는다.

[추정] 프로젝트는 활동 저조를 사회적 고립 예방을 위한 점검 신호로 삼고, 위험과 생활권 인프라 접근성을 결합해 행정 담당자의 예방 개입을 지원한다. 개인의 사회적 고립 여부를 진단하거나 실제 개입의 인과효과를 검증한 분석은 아니다.

[확실] 분석 대상은 강남구·춘천시의 격자·연령 집단이다. 사건은 활동 저조 선정 상태의 첫 진입이며, 시설 자료의 원시 가중합은 중력 기반 접근성 지표다. LLM 에이전트는 설계·시연 수준이고 이 노트북은 API를 호출하지 않는다.

활동 잔차·선정 결과 → 시설 질량과 α 보정 중력 X → 기존 생존 라벨 → 저장된 AFT 모델·평가 → 단기 위험·우선 대상 → 대응 유형·거점·조건부 서비스 연결

## 실행 환경과 데이터 경로

[확실] Python, NumPy, pandas, Matplotlib, IPython을 사용한다. 외부 패키지를 사용하지 않는 독립 엔진이라고 표현하지 않는다. 경로가 달라졌다면 아래 MAIN과 WORK 또는 P의 개별 항목을 수정한다.

[확실] 이름이 같은 첫 번째 파일을 재귀 검색하는 대신 경로를 지정한다. 파일이 없으면 중단한다. 아래 경로 목록과 SHA-256 해시를 통해 어떤 버전의 데이터를 읽었는지 확인할 수 있다.

In [ ]:
from pathlib import Path
import json, hashlib, math
from html import escape
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

MAIN = Path(r'./private_inputs/main_work')
WORK = Path(r'./private_inputs/workspace')
BASE = MAIN / 'monthly_grid_results'
ALPHA = MAIN / 'initial_pipeline_alpha_results/results'
STEP5 = MAIN / 'step5_alpha_results/results'
STEP6_12 = WORK / 'step6_parts12/results'
STEP6_345 = WORK / 'step6_parts345'
P = {
    'scores': BASE / '20261025_195726_925731/04_all_scores.csv',
    'labels': BASE / 'step3_survival_results/results/patient_survival_interval_labels.csv',
    'label_meta': BASE / 'step3_survival_results/results/run_metadata.json',
    'base_mass': BASE / 'step4_aft_and_3d_gravity/inputs/mass_all_ages.csv',
    'alpha_mass': ALPHA / 'mass_all_ages_alpha.csv',
    'alpha_meta': ALPHA / 'run_metadata.json',
    'coefficients': ALPHA / 'selected_full_refit_coefficients.csv',
    'models': ALPHA / 'full_refit_models.json',
    'evaluation': ALPHA / 'heldout_test_metrics.csv',
    'priority': STEP5 / 'high_risk_intervention_prescriptions.csv',
    'priority_thresholds': STEP5 / 'priority_thresholds.csv',
    'step5_meta': STEP5 / 'run_metadata.json',
    'routes': STEP6_12 / 'priority_response_classification.csv',
    'facility_candidates': STEP6_12 / 'facility_connection_candidates.csv',
    'catalog': STEP6_345 / 'data/official_services.json',
    'links': STEP6_345 / 'results/conditional_service_links.csv',
    'cases': STEP6_345 / 'results/illustrative_agent_cases.json',
    'site_comparison': STEP6_345 / 'results/site_scenario_comparison.csv',
}
missing = [str(p) for p in P.values() if not p.is_file()]
if missing:
    raise FileNotFoundError('입력 경로를 확인하세요:\n' + '\n'.join(missing))
manifest = pd.DataFrame([
    {'input': k, 'path': str(p), 'bytes': p.stat().st_size,
     'sha256': hashlib.sha256(p.read_bytes()).hexdigest()}
    for k, p in P.items()
])
display(manifest)
plt.rcParams.update({'font.family': 'Malgun Gothic', 'axes.unicode_minus': False,
                     'figure.dpi': 110, 'font.size': 10})
AGES = ['20G', '30G', '40G', '50G', '60GU']
AGE_NAMES = {'20G':'20대', '30G':'30대', '40G':'40대', '50G':'50대', '60GU':'60대 이상'}
def read_table(key):
    return pd.read_csv(P[key], dtype={'GRID_ID': str, 'facility_id': str})
def read_json(key):
    return json.loads(P[key].read_text(encoding='utf-8-sig'))

## Step0 활동 잔차와 활동 저조 선정

[확실] 활동 잔차는 실제 유동인구와 비교 기준 유동인구의 log1p 차이로 구성한 기존 분석 결과다. 비교 기준은 유사 환경 격자의 유동인구를 반영한 값이며, 개인의 외출 횟수·고립 상태가 아니다.

$$r_i = \log(1+\mathrm{FLOW}_i)-\log(1+\mathrm{PEER\_MEDIAN}_i)$$

[확실] 선정은 실제 저장된 SELECTED를 사용한다. 연령별 비율은 Step3 메타데이터의 quota를 표시한다. 하위20% 또는 잔차 -1이라는 기준을 새로 적용하지 않는다. 이 비율은 개인 고립의 공식 진단 기준이 아니다.

In [ ]:
cols = ['STD_YM','GRID_ID','REGION','DONG','AGE','FLOW','ACTIVITY_RESIDUAL','SELECTED']
scores = pd.read_csv(P['scores'], usecols=cols, dtype={'GRID_ID':str})
assert not scores.duplicated(['STD_YM','GRID_ID','AGE']).any()
label_meta = read_json('label_meta')
quota = pd.Series(label_meta['quota'], name='선정비율').rename_axis('AGE').reset_index()
display(quota)
print(f'[확실] 전체 관측 행: {len(scores):,}개 격자·연령·월 조합')
selection = scores.groupby(['STD_YM','REGION','AGE']).agg(
    관측조합수=('GRID_ID','size'), 선정조합수=('SELECTED','sum'), 선정비율=('SELECTED','mean')
).reset_index()
display(selection.head(15))
fig, ax = plt.subplots(figsize=(9, 4))
for region, g in scores.groupby('REGION'):
    trend = g.groupby('STD_YM').ACTIVITY_RESIDUAL.median()
    ax.plot(trend.index.astype(str), trend.values, marker='o', label=region)
ax.set(title='지역별 활동 잔차 중앙값', xlabel='기준월', ylabel='활동 잔차 중앙값')
ax.legend(); fig.tight_layout(); plt.show()

## Step1과 Step2 시설 질량과 중력 기반 접근성

[확실] 기존 시설 질량 M은 카드 집계·주변 접점 등의 대리 지표를 사용한 값이다. 개별 시설의 실매출·실제 이용량·수용능력으로 해석하지 않는다. 기존 M_60은 카드 60대와 연결되어 SK의 60GU 집계와 연령 범위가 완전히 같지 않다.

[확실] α 보정은 7월 카드 데이터1의 업종·연령별 이용 시간 및 요일 분포와 시설 주변의 유동인구를 결합했다. 연령×시간대의 개인 단위 교차 관측치를 만든 것은 아니다. 검증 표본에서 반경800m와 강도 -1이 선택됐다.

$$\alpha_{j,a}=\frac{\exp(-z_{j,a})}{\mathrm{mean}_{지역,종류,연령}[\exp(-z)]}$$
$$M^*_{j,a}=\alpha_{j,a}M_{j,a},\qquad G_{i,k,a}=\sum_{j\in k}\frac{M^*_{j,a}}{\max(d_{ij},1\mathrm{m})^2}$$

[확실] 실제 모델 입력 X는 위 원시 중력 G를 기존 log1p·정규화 기준으로 변환한 값이다. 목적지 간 매력도로 나누는 허프 선택확률은 계산하지 않았다. α는 질량 가중 평균이 아니라 그룹별 단순 평균을 1로 맞췄다.

In [ ]:
base_mass = read_table('base_mass')
alpha_mass = read_table('alpha_mass')
alpha_meta = read_json('alpha_meta')
print('[확실] 선택된 보정 후보:', alpha_meta['selected'])
display(alpha_mass.groupby(['region','카테고리']).size().rename('시설수').reset_index())
display(alpha_mass[['facility_id','명칭','카테고리','region','M_20','M_30','M_40','M_50','M_60']].head())
assert not alpha_mass.facility_id.duplicated().any()

## Step3 사건과 검열

[확실] 기준 시점은 2025년7월(t=0), 마지막 관측은 12월(t=5)이다. 여섯 월별 관측 시점이며, 기준 이후 시간 간격은 5개월이다. 7월 미선정 집단을 추적하고 처음 SELECTED가 참으로 관측되는 월 구간을 사건으로 정의한다. 미선정은 정상 활동 판정이 아니다.

[확실] 사건 시점은 월별 관측 사이의 $(L,U]$로 표현한다. 사건 없이 관측이 끝나거나 결측으로 추적을 중단한 경우는 우측검열이다. 검열 이후의 상태를 임의로 이어 붙이지 않는다.

In [ ]:
labels = read_table('labels')
assert not labels.duplicated(['GRID_ID','AGE']).any()
print(f'[확실] 생존 분석 표본: {len(labels):,}개 격자·연령 조합')
display(labels.groupby(['REGION','AGE','status']).size().rename('조합수').reset_index())
observed = labels.groupby('AGE').event_observed.agg(['size','sum','mean']).reindex(AGES)
observed.columns = ['분석조합수','첫진입관측수','첫진입관측비율']
display(observed)
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.bar([AGE_NAMES[a] for a in observed.index], observed.첫진입관측비율*100, color='#267D8D')
ax.set(title='7~12월 관측 중 활동 저조 첫 진입 관측 비율', ylabel='관측 비율 (%)')
fig.tight_layout(); plt.show()
print('[확실] 위 비율은 사건 관측 플래그의 단순 평균이며 검열을 보정한 누적발생확률이 아닙니다.')

## Step4 저장된 AFT 모델과 평가 결과

[확실] Weibull AFT는 중력 기반 X와 사건 시점의 연관을 추정하는 데 사용했다. 중력모델은 공간적 접근성을 구성하고 AFT는 시점·조건부 확률을 계산한다.

$$S(t|x)=\exp[-(t\exp(-\eta))^\rho],\quad\eta=\beta_0+x^T\beta$$

[확실] 양의 회귀계수는 다른 입력을 고정했을 때 모델상 사건 시점이 늦어지는 방향을 뜻한다. 시설 확충의 실제 보호 효과나 인과효과로 해석하지 않는다.

[확실] 아래 셀은 저장된 최종 계수와 평가 결과를 읽는다. 이 노트북에서 AFT를 재학습하지 않는다. 보고된 성능은 500m 공간 셀 분리 평가이며 미래 기간 검증은 아니다. 기존 전체기간 질량과 기존 라벨 구성도 보존했다.

In [ ]:
coeff = read_table('coefficients')
models = read_json('models')
evaluation = read_table('evaluation')
primary = evaluation[evaluation['split'].eq('test')]
comparison = primary.pivot(index='AGE', columns='candidate', values=['auc_5m','interval_nll','brier_5m']).reindex(AGES)
display(comparison)
print('[확실] 모델별 연령 단순 평균')
display(primary.groupby('candidate')[['auc_5m','interval_nll','brier_5m']].mean())
coef_view = coeff[coeff.term.str.startswith('X_')].pivot(index='term',columns='AGE',values='beta').reindex(columns=AGES)
display(coef_view)
fig, ax = plt.subplots(figsize=(9, 3.6))
v = coef_view.to_numpy(); bound = np.nanmax(np.abs(v))
im = ax.imshow(v, cmap='RdBu_r', vmin=-bound, vmax=bound, aspect='auto')
ax.set_xticks(range(5), [AGE_NAMES[a] for a in AGES]); ax.set_yticks(range(len(coef_view)), coef_view.index)
ax.set_title('활동 저조 첫 진입 시점과의 연관 방향과 AFT 계수')
fig.colorbar(im, ax=ax, label='회귀계수 beta'); fig.tight_layout(); plt.show()
print('[확실] 현재 비교는 기존 중력 X와 α 보정 중력 X입니다. 활동 잔차 단독 대비 추가 기여를 보는 A/B/C 비교는 포함하지 않았습니다.')

## Step5 조건부 잔여시간과 단기 위험

[확실] 골든타임은 업무상 이름이며 계산값은 사건 미발생 조건의 잔여시간 중앙값이다. 실제 확보한 예방 기간이나 검증된 개입 마감기한이 아니다.

$$q_{0.5}(t)=[t^\rho+\log(2)\exp(\rho\eta)]^{1/\rho}-t$$
$$P(t<T\le t+h|T>t)=1-\exp\{-[(t+h)^\rho-t^\rho]\exp(-\rho\eta)\}$$

[확실] 12월의 사건 미발생·예측 가능 대상 중 지역·연령별 잔여시간 하위5%를 우선 대상으로 선정했다. ‘확률35% 이상 또는 3개월 이하’라는 규칙이나 공식 선정 목표는 사용하지 않았다. 아래 수치는 파일에서 다시 계산한다.

In [ ]:
def remaining_median(eta, rho, t):
    eta, rho, t = np.broadcast_arrays(np.asarray(eta,float), np.asarray(rho,float), np.asarray(t,float))
    log_t = np.log(np.maximum(t,1e-300))
    log_q = np.logaddexp(np.where(t>0,rho*log_t,-np.inf), rho*eta+np.log(np.log(2.0)))/rho
    return np.where(t>0,t*np.expm1(np.maximum(log_q-log_t,0)),np.exp(log_q))

def conditional_probability(eta, rho, t, h):
    eta,rho,t,h = np.broadcast_arrays(np.asarray(eta,float),np.asarray(rho,float),np.asarray(t,float),np.asarray(h,float))
    end=t+h
    diff=np.where(t>0,end**rho*(-np.expm1(rho*np.log(np.maximum(t,1e-300)/end))),end**rho)
    return -np.expm1(-np.exp(np.clip(np.log(np.maximum(diff,1e-300))-rho*eta,-745,700)))

priority = read_table('priority')
thresholds = read_table('priority_thresholds')
step5_meta = read_json('step5_meta')
print(f'[확실] 기준월 {step5_meta["latest_month"]}; 우선 대상 {len(priority):,}개 격자·연령 조합; 고유 격자 {priority.GRID_ID.nunique():,}개')
display(thresholds)
p1 = conditional_probability(priority.eta, priority['shape'], priority.elapsed_months, 1.)
q50 = remaining_median(priority.eta, priority['shape'], priority.elapsed_months)
assert np.allclose(p1, priority.event_probability_next_1m, atol=1e-10)
assert np.allclose(q50, priority.remaining_median_months, rtol=1e-10)
assert np.allclose(conditional_probability(priority.eta,priority['shape'],priority.elapsed_months,q50),.5,atol=1e-10)
print('[확실] 保存 확률·잔여시간 재현 및 중앙값의 조건부 확률 0.5 점검 통과'.replace('保存','저장된'))
summary = priority.groupby(['REGION','AGE']).agg(
    우선대상조합수=('GRID_ID','size'),
    일개월위험중앙값=('event_probability_next_1m','median'),
    이개월위험중앙값=('event_probability_next_2m','median'),
    모델잔여시간중앙값=('remaining_median_months','median')
).reset_index()
display(summary)

[확실] 12월 이후1~2개월은 2026년1~2월에 해당하며 실제 해당 기간 결과로 검증하지 않았다. 긴 잔여시간도 외삽한 모형값이다. 2026년10월의 현재 위험으로 표시하지 않는다. 지역별 우선5%는 절대 위험 기준을 충족한 인원을 뜻하지 않는다.

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(12,3.8))
for region,g in priority.groupby('REGION'):
    axes[0].hist(g.event_probability_next_1m, bins=20, alpha=.55, label=region)
axes[0].set(title='우선 대상의 1개월 활동 저조 진입 확률', xlabel='모델 조건부 확률', ylabel='격자·연령 조합 수')
axes[0].legend()
axes[1].boxplot([priority[priority.AGE.eq(a)].remaining_median_months for a in AGES], tick_labels=[AGE_NAMES[a] for a in AGES])
axes[1].set(title='우선 대상의 모델상 잔여시간 분포', ylabel='잔여시간 중앙값 (개월)')
fig.tight_layout(); plt.show()

## Step6 대응 유형과 거점 후보

[확실] 시설 종류별 중력 하위25%는 같은 지역·연령의 전체 모델 격자와 비교한 운영 기준이다. 의료·버스·도서관·공원·사교의 공통5종 중 하나 이상 낮으면 VISIT_OR_MOBILITY_FIRST, 해당 항목이 없으면 EXISTING_RESOURCE_LINK_FIRST로 분류했다. 둘째 유형을 모든 자원이 충분한 지역으로 단정하지 않는다.

[추정] 첫째 유형에서는 이동·외출 제약과 방문 지원 의향을 확인한다. 둘째 유형에서는 기존 거점의 이용 경험·신청 장벽·참여 의향을 확인한다. 어느 유형도 개인의 이동 능력이나 고립 원인을 확정하지 않는다.

In [ ]:
routes = read_table('routes')
candidates = read_table('facility_candidates')
assert not routes.duplicated(['GRID_ID','AGE']).any()
assert set(routes.route_code).issubset({'VISIT_OR_MOBILITY_FIRST','EXISTING_RESOURCE_LINK_FIRST','EXPOSURE_REVIEW_REQUIRED'})
display(routes.groupby(['REGION','route_code']).size().rename('격자연령조합수').reset_index())
print(f'[확실] 시설 후보 연결 {len(candidates):,}건; 중복 제거 후보 시설 {candidates.facility_id.nunique():,}곳')
display(candidates[['GRID_ID','AGE','category','facility_name','address','distance_m','gravity_rank','is_nearest','category_contribution_share']].head(12))
print('[확실] 거리는 직선거리이며, 상위 기여 시설과 최단거리 시설은 다를 수 있습니다. 시설이 해당 복지사업 제공기관인지 별도 확인해야 합니다.')

## Step6 공식 서비스와 조건부 연결

[확실] 아래 카탈로그는 공식 서비스 안내 7종이며 조례7종이 아니다. 지역·연령 범위에 맞는 조건부 후보를 생성했을 뿐 개인의 자격·돌봄 필요·소득·이용 동의·제공 여력을 확인한 것은 아니다. 60GU 집계는 65세 이상 서비스의 자동 자격이 아니다.

[확실] 카탈로그에는 춘천 일상돌봄·노인맞춤돌봄·의료요양 통합돌봄, 서울 고립가구 전담기구 연계·고립은둔청년 지원, 강남 방문의료·통합돌봄이 들어 있다. 확인일은2026년10월26일이며 2025년 당시 이용 가능성을 소급 검증한 자료는 아니다.

In [ ]:
services = read_json('catalog')
links = read_table('links')
display(pd.DataFrame(services)[['service_id','name','region','age_min','age_max','eligibility_note','application_entry','source_url','verified_on']])
print(f'[확실] 서비스 {len(services)}종; 조건부 연결 {len(links):,}건')
assert links.eligibility_status.eq('CONDITIONAL_CANDIDATE_NOT_APPROVED').all()
display(links[['GRID_ID','AGE','service_name','age_check','unverified_conditions','eligibility_status']].head(10))

## Step6 실제 격자 기반 에이전트 활용 사례

[확실] 다음4개는 지역과 대응 유형을 조합해 선택한 실제 격자 사례다. 개인 페르소나나 상담 결과가 아니다. 연령은 20대3개·60대 이상1개이며 독거·질병·복합위기가구라는 개인 특성을 관측하지 않았다.

[추정] 에이전트는 위험·중력·시설·서비스 조건을 조회하고 확인할 질문과 담당자의 다음 행동을 작성하도록 설계했다. 아래 출력은 설계 시연 예시로 실제 LLM 응답이나 자동 신청 결과가 아니다.

In [ ]:
cases = read_json('cases')
for item in cases:
    ctx=item['observed_grid_context']
    print('\n'+'='*70)
    print(f'[확실] {item["case_id"]}: {ctx["REGION"]} {ctx["DONG"]} / {AGE_NAMES[ctx["AGE"]]}')
    print('[확실] 격자:',ctx['GRID_ID'])
    print(f'[확실] 모델 1개월 {ctx["event_probability_next_1m"]:.2%}, 2개월 {ctx["event_probability_next_2m"]:.2%}')
    print('[확실] 대응 분류:',ctx['route_code'])
    print('[확실] 상대적으로 낮은 종류:',item['low_categories'] or '공통5종 하위25% 항목 없음')
    print('[추정] 설계 시연 대응안:',item['illustrative_agent_output'])
    print('[확실] 개인 상황:',item['field_input_status'])
    display(pd.DataFrame(item['conditional_services'])[['service_name','age_check','unverified_conditions','source_url']])

## Step6 실제 위치 기반 가상 거점 비교

[확실] 24개 기존 시설 위치에 가상의 찾아가는 건강·복지 접점을 추가하는 배치안을 비교했다. 신규 질량은 지역·연령별 기존 의료 보정 질량 중앙값1개분으로 고정했다. 실제 제공 인력·진료 능력을 측정한 값이 아니며 사용 허가·운영기관도 미확인이다.

[확실] 순위 기준은 의료 중력이 낮은 우선 대상 중 원시 의료 접근성이10% 이상 증가한 격자·연령 조합 수다. 전체 가능한 후보지의 최적해나 실제 예방 인원을 의미하지 않는다. 위험확률 변화는 고정된 AFT 계수의 시나리오 반응이다.

In [ ]:
sites = read_table('site_comparison')
display(sites.groupby('REGION').head(3)[[
    'REGION','site_name','address','n_medical_low_gain10pct',
    'unique_low_grids_gain10pct','n_medical_low_cross_q25','region_rank'
]])

## 결과의 활용과 운영 시 확인할 항목

[확실] 완료한 작업은 활동 저조 선정·기존 라벨 기반 AFT·중력 보정 비교·우선 대상 선정·거점 후보 분해·공식 서비스 조건부 연결·실제 위치 기반 가상 시나리오다.

[추정] 행정 담당자는 우선 점검 지역을 선택하고 연결 공백 또는 이용 장벽을 확인한 뒤, 자격과 의향을 대조해 적합한 기관에 연결한다. 운영 시에는 연락 성공률·신청률·실제 이용률·연결 소요시간·미이용 사유 등을 평가할 수 있다.

[확실] 평균2~4개월 선제 개입, 비용 절감, 실제 고립 예방 인원, LLM 판단 정확도, 서비스100% 적합성은 검증한 성과가 아니다. 통계적·인과적 정합성을 완비했다거나 오차 없이 적합했다고 주장하지 않는다. 발표에는 활동 저조의 모델 결과와 사회적 고립 예방이라는 활용 목적을 구분한다.